# CS781 Phase 5 — Notebook P5_01: Statute-identification data, statistics, tokenisation, baselines

No neural network is trained here. The notebook prepares the LSI data once (so every P5_02 experiment starts in
seconds) and answers the questions that shape the modelling: how many statutes, how rare are they, how many per
document, which ones co-occur, which part of the document carries the signal, and **how much the decision rule
(thresholds) matters for macro-F1**.

| Step | What happens |
|---|---|
| 1 | download `Phase_5_LSI` (≈ 480 MB) from Hugging Face, parse id / text / statute-ID list (test labels are never read) |
| 2 | checks: valid label ids, repeated ids, exact duplicates, **overlap between train / dev / test** |
| 3 | statistics: statutes per document, frequency of every statute (frequent / medium / rare thirds), co-occurrence, lengths, casing |
| 4 | explicit statute mentions in the text ("section 302", "u/s 34", …) and which number each statute ID corresponds to |
| 5 | tokenise every document with the Phase-1 tokenizer → compact `uint16` token file per split + offsets |
| 6 | baselines with the official metric: frequency prior, TF-IDF + one-vs-rest logistic regression on the first / last 512 words and the whole document, under 5 decision rules |
| 7 | save everything P5_02 needs |

**Kaggle settings**: Accelerator **None**, Internet **ON**, secret `HF`. Run time ≈ 30–45 min (most of it the
3 × 100 logistic regressions). **Input**: the **NB1 output** (only `tokenizer/bpe32768/tokenizer.json`).
**Output**: `/kaggle/working/cs781_p1/p5/data/…` → attach this notebook's output to P5_02.

In [1]:
!pip -q install tokenizers

In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1146.0 GB / total 8758.4 GB
HF token found: True


## 0. Configuration and download

In [3]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 120)
try: display
except NameError: display = print

P5_DATA = ROOT / "p5" / "data"; P5_DATA.mkdir(parents=True, exist_ok=True)
P5_REPO_DIR = "Phase_5_LSI"
P5_FILES = {"train": "Train/train.json", "dev": "Train/dev.json", "test": "Test/test.json"}
TOKENIZER = "bpe32768"
SECTION_WORDS = 512                 # size of the head / tail sections for the TF-IDF baselines
BASELINE_SECTIONS = ("head", "tail", "full")
P5_LOCAL = os.environ.get("CS781_P5_LOCAL")
N_JOBS = os.cpu_count() or 2

def get_p5_file(rel):
    if P5_LOCAL: return Path(P5_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P5_REPO_DIR}/{rel}", repo_type="dataset",
                                token=HF_TOKEN, local_dir=str(SCRATCH / "p5raw")))

PATHS = {k: get_p5_file(v) for k, v in P5_FILES.items()}
for k, p in PATHS.items(): print(f"{k}: {p}  ({os.path.getsize(p)/1e6:.0f} MB)")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

train: /tmp/cs781_scratch/p5raw/Phase_5_LSI/Train/train.json  (299 MB)
dev: /tmp/cs781_scratch/p5raw/Phase_5_LSI/Train/dev.json  (79 MB)
test: /tmp/cs781_scratch/p5raw/Phase_5_LSI/Test/test.json  (98 MB)


## 1. Parse
Accepts a JSON list of records or a dict `{id: record}`, the usual field names (`id`/`case_id`, `text`/`document`,
`labels`/`label`/`statutes`/`gold`), labels as a list of ints (or numeric strings), a text stored as a list of strings.
Ids are kept as strings (the evaluator requires string ids). **Test labels are never read.**

In [4]:
ID_KEYS = ("id", "case_id", "name", "doc_id", "uid")
TEXT_KEYS = ("text", "document", "case_text", "judgment", "facts")
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "gold", "y", "target")

def first_key(r, keys):
    return next((k for k in keys if k in r), None)

def to_label_list(lab):
    if lab is None: return []
    if not isinstance(lab, (list, tuple)): lab = [lab]
    out = []
    for x in lab:
        assert not isinstance(x, bool) and (isinstance(x, int) or (isinstance(x, str) and x.strip().lstrip("-").isdigit())), \
            f"unexpected label value {x!r} — labels must be integer statute ids"
        out.append(int(x))
    return out

def parse(obj, split):
    recs = list(obj.items()) if isinstance(obj, dict) else [(None, r) for r in obj]
    sample = recs[0][1] if recs else {}
    print(f"{split}: {len(recs)} records; record type {type(sample).__name__}; keys {list(sample)[:10] if isinstance(sample, dict) else '-'}")
    ids, texts, labels = [], [], []
    for k, r in recs:
        if not isinstance(r, dict): r = {"text": r}
        idk, tk = first_key(r, ID_KEYS), first_key(r, TEXT_KEYS)
        t = r[tk] if tk else ""
        if isinstance(t, list): t = " ".join(map(str, t))
        ids.append(str(r[idk]) if idk else str(k)); texts.append(str(t))
        if split != "test":                                   # test labels are private: never read them
            labels.append(to_label_list(r.get(first_key(r, LABEL_KEYS))))
    return ids, texts, (labels if split != "test" else None)

DATA = {}
for split, p in PATHS.items():
    t0 = time.time()
    with open(p, encoding="utf-8") as f: obj = json.load(f)
    DATA[split] = dict(zip(("ids", "texts", "labels"), parse(obj, split)))
    del obj; gc.collect()
    print(f"   parsed in {time.time()-t0:.0f}s; first text: {DATA[split]['texts'][0][:300]!r}")
    if DATA[split]["labels"] is not None: print(f"   first labels: {DATA[split]['labels'][:3]}")

train: 42750 records; record type dict; keys ['id', 'text', 'labels']
   parsed in 4s; first text: '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of industry, commerceand trade in and around the city of Mumbai which generatessubstantial quantity of wealth, there has been increase oforganised activiti'
   first labels: [[69, 9, 3], [81, 5, 94], [10, 17, 88, 7]]
dev: 10181 records; record type dict; keys ['id', 'text', 'labels']
   parsed in 1s; first text: 'Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the prosecution relied on the evidence of many persons who supposedly witnessedthe occurrence, while deposing in Court, most of them resiled from the '
   first labels: [[5], [58, 55, 5], [41, 5, 61]]
test: 13019 records; record type dict; keys ['id', 'text']
   parsed in 1s; first text: '

## 2. Checks: label ids, repeated ids, duplicates, overlap between splits
* every train/dev document must have ≥ 1 valid (non-negative, unique) statute id;
* the label space is `0 … max id` (the evaluator derives it from the reference as max gold id + 1, and **rejects any
  predicted id ≥ that** — so predicting an id that never occurs in the test reference would make a submission fail);
* exact duplicate texts in train (same text, different statute sets are merged as the union);
* texts shared between train / dev / test (Phase 4's dev file contained every test document).

In [5]:
for split, d in DATA.items():
    if d["labels"] is not None:
        bad = [i for i, l in enumerate(d["labels"]) if any(x < 0 for x in l) or len(set(l)) != len(l)]
        empty = sum(1 for l in d["labels"] if not l)
        print(f"{split}: docs with negative/duplicate ids: {len(bad)} | docs without any statute: {empty}")
        d["labels"] = [sorted(set(l)) for l in d["labels"]]
    n_rep = len(d["ids"]) - len(set(d["ids"]))
    if n_rep:
        assert split != "test", "test ids are not unique"
        first, keep, same_txt, diff_txt = {}, [], 0, 0
        for i, (did, t) in enumerate(zip(d["ids"], d["texts"])):
            if did not in first: first[did] = [i]; keep.append(i); continue
            if any(" ".join(d["texts"][j].split()) == " ".join(t.split()) for j in first[did]):
                same_txt += 1; j0 = first[did][0]; d["labels"][j0] = sorted(set(d["labels"][j0]) | set(d["labels"][i])); continue
            diff_txt += 1; first[did].append(i); d["ids"][i] = f"{did}#{len(first[did])}"; keep.append(i)
        for k in ("ids", "texts", "labels"): d[k] = [d[k][i] for i in keep]
        print(f"{split}: {n_rep} repeated ids -> {same_txt} exact copies merged, {diff_txt} different texts kept under a new id")
    assert len(set(d["ids"])) == len(d["ids"])
    print(f"{split}: {len(d['ids'])} docs, ids unique, empty texts: {sum(not t.strip() for t in d['texts'])}")

def h(t): return hash64(" ".join(t.lower().split()))
HASH = {s: [h(t) for t in d["texts"]] for s, d in DATA.items()}
groups = defaultdict(list)
for i, hh in enumerate(HASH["train"]): groups[hh].append(i)
dups = {hh: g for hh, g in groups.items() if len(g) > 1}
conflict = sum(1 for g in dups.values() if len({tuple(DATA["train"]["labels"][i]) for i in g}) > 1)
print(f"train: {sum(len(g) - 1 for g in dups.values())} redundant duplicate docs in {len(dups)} groups; {conflict} groups with different statute sets (merged as union)")
keep = []
for hh, g in groups.items():
    if len(g) > 1: DATA["train"]["labels"][g[0]] = sorted(set().union(*[DATA["train"]["labels"][i] for i in g]))
    keep.append(g[0])
keep.sort()
if len(keep) < len(DATA["train"]["ids"]):
    for k in ("ids", "texts", "labels"): DATA["train"][k] = [DATA["train"][k][i] for i in keep]
    HASH["train"] = [HASH["train"][i] for i in keep]
print("train docs after de-duplication:", len(DATA["train"]["ids"]))

OVERLAP = {"dev_in_train": int(sum(x in set(HASH["train"]) for x in HASH["dev"])),
           "test_in_train": int(sum(x in set(HASH["train"]) for x in HASH["test"])),
           "test_in_dev": int(sum(x in set(HASH["dev"]) for x in HASH["test"]))}
print("identical texts across splits:", OVERLAP)
print("id overlap train/dev:", len(set(DATA["train"]["ids"]) & set(DATA["dev"]["ids"])),
      "| train/test:", len(set(DATA["train"]["ids"]) & set(DATA["test"]["ids"])), "| dev/test:", len(set(DATA["dev"]["ids"]) & set(DATA["test"]["ids"])))
if OVERLAP["test_in_dev"] or OVERLAP["test_in_train"]:
    print("!! some test documents also appear (with labels) in train/dev — P5_02 must not tune on / train on those copies")

train: docs with negative/duplicate ids: 3 | docs without any statute: 0
train: 42750 docs, ids unique, empty texts: 0
dev: docs with negative/duplicate ids: 1 | docs without any statute: 0
dev: 10181 docs, ids unique, empty texts: 0
test: 13019 docs, ids unique, empty texts: 0
train: 435 redundant duplicate docs in 256 groups; 80 groups with different statute sets (merged as union)
train docs after de-duplication: 42315
identical texts across splits: {'dev_in_train': 118, 'test_in_train': 153, 'test_in_dev': 98}
id overlap train/dev: 0 | train/test: 0 | dev/test: 0
!! some test documents also appear (with labels) in train/dev — P5_02 must not tune on / train on those copies


## 3. Statistics: label space, statutes per document, frequency, co-occurrence, length, casing
**Macro-F1 gives every statute the same weight**, so the rare ones matter as much as the frequent ones. The label ids
are split into three equal groups by training frequency (frequent / medium / rare); later notebooks report F1 per group.

In [6]:
N_LABELS = 1 + max(max((max(l) for l in DATA[s]["labels"] if l), default=-1) for s in ("train", "dev"))
def multihot(labels, n=N_LABELS):
    Y = np.zeros((len(labels), n), np.uint8)
    for i, l in enumerate(labels): Y[i, l] = 1
    return Y
Y = {s: multihot(DATA[s]["labels"]) for s in ("train", "dev")}
cnt_tr, cnt_dv = Y["train"].sum(0), Y["dev"].sum(0)
print(f"label space: 0 … {N_LABELS - 1} ({N_LABELS} statute ids) | ids never in train: {np.where(cnt_tr == 0)[0].tolist()} | never in dev: {np.where(cnt_dv == 0)[0].tolist()}")
card = {s: Y[s].sum(1) for s in Y}
rows = [{"split": s, "docs": len(card[s]), "statutes/doc mean": card[s].mean(), "median": np.median(card[s]), "max": card[s].max(),
         "share with 1": (card[s] == 1).mean(), "share with ≥5": (card[s] >= 5).mean(), "distinct statute sets": len({tuple(l) for l in DATA[s]["labels"]})}
        for s in Y]
display(pd.DataFrame(rows).set_index("split").round(3))
print("statutes per document (train):", dict(sorted(Counter(card["train"].tolist()).items())))

order = np.argsort(-cnt_tr)
BUCKET = np.empty(N_LABELS, dtype=object)
for b, part in zip(("frequent", "medium", "rare"), np.array_split(order, 3)): BUCKET[part] = b
lab_df = pd.DataFrame({"label": np.arange(N_LABELS), "train_docs": cnt_tr, "train_share": cnt_tr / len(Y["train"]),
                       "dev_docs": cnt_dv, "bucket": BUCKET}).sort_values("train_docs", ascending=False)
print("\nper-bucket training-document counts:")
display(lab_df.groupby("bucket")["train_docs"].describe()[["count", "min", "50%", "max"]])
print("most frequent 15 and rarest 15 statute ids:")
display(pd.concat([lab_df.head(15), lab_df.tail(15)]).set_index("label").round(4))
print(f"imbalance: most frequent statute in {cnt_tr.max()} docs, rarest in {cnt_tr.min()} — ratio {cnt_tr.max() / max(1, cnt_tr.min()):.0f}x; "
      f"positives are {Y['train'].mean():.2%} of all (document, statute) pairs")

# co-occurrence: counts, conditional probability P(b | a) and lift = P(a,b) / (P(a) P(b))
C = Y["train"].T.astype(np.int64) @ Y["train"].astype(np.int64)
np.fill_diagonal(C, 0); n_tr = len(Y["train"])
pairs = [(a, b, C[a, b]) for a in range(N_LABELS) for b in range(a + 1, N_LABELS) if C[a, b] > 0]
co = pd.DataFrame(pairs, columns=["a", "b", "both"])
co["P(b|a)"] = co["both"] / cnt_tr[co["a"]]; co["P(a|b)"] = co["both"] / cnt_tr[co["b"]]
co["lift"] = co["both"] * n_tr / (cnt_tr[co["a"]] * cnt_tr[co["b"]])
print(f"\nco-occurring statute pairs: {len(co)} of {N_LABELS * (N_LABELS - 1) // 2} possible")
print("most frequent pairs:"); display(co.sort_values("both", ascending=False).head(12).round(3).to_string(index=False))
print("strongest associations (lift, pairs in ≥ 30 docs):"); display(co[co["both"] >= 30].sort_values("lift", ascending=False).head(12).round(3).to_string(index=False))
hub = (C > 0).sum(1); print("statutes that co-occur with the most others (hubs):", [(int(i), int(hub[i])) for i in np.argsort(-hub)[:5]])

rows = []
for s, d in DATA.items():
    w = np.array([len(t.split()) for t in d["texts"]]); d["words"] = w
    rows.append({"split": s, "docs": len(w), "words_mean": w.mean(), "words_median": np.median(w), "words_p90": np.percentile(w, 90), "words_max": w.max()})
display(pd.DataFrame(rows).set_index("split").round(1))
if True:   # do longer documents cite more statutes?
    q = np.quantile(DATA["train"]["words"], [0, .25, .5, .75, 1])
    print("statutes/doc by length quartile (train):", [round(float(card["train"][(DATA["train"]["words"] >= q[j]) & (DATA["train"]["words"] <= q[j + 1])].mean()), 2) for j in range(4)])

def is_lower(t): s = t[:5000]; return sum(c.isupper() for c in s) < 0.002 * max(1, len(s))
LOWER_SHARE = {s: float(np.mean([is_lower(t) for t in d["texts"]])) for s, d in DATA.items()}
print("share of documents that are lower-case:", {k: round(v, 3) for k, v in LOWER_SHARE.items()})
LOWERCASE = "auto"     # lower-case every split if the test documents are (almost) all lower-case
DO_LOWER = (LOWER_SHARE["test"] > 0.9 and min(LOWER_SHARE.values()) < 0.9) if LOWERCASE == "auto" else bool(LOWERCASE)
print("=> lower-case all splits:", DO_LOWER)
if DO_LOWER:
    for d in DATA.values(): d["texts"] = [t.lower() for t in d["texts"]]
masks = Counter(m for t in DATA["train"]["texts"][:3000] for m in re.findall(r"[<\[][A-Za-z_]{2,20}[>\]]", t))
print("placeholder tokens in the first 3000 training docs (masked entities):", masks.most_common(10))
print("\nexample (dev, first 700 chars):\n", DATA["dev"]["texts"][0][:700])

label space: 0 … 99 (100 statute ids) | ids never in train: [] | never in dev: []


,docs,statutes/doc mean,median,max,share with 1,share with ≥5,distinct statute sets
split,,,,,,,
train,42315,3.710,3.0,37,0.214,0.308,16138
dev,10181,3.916,3.0,28,0.152,0.332,5415


statutes per document (train): {1: 9054, 2: 7520, 3: 6983, 4: 5708, 5: 4206, 6: 3254, 7: 2099, 8: 1375, 9: 821, 10: 451, 11: 356, 12: 165, 13: 113, 14: 56, 15: 47, 16: 27, 17: 17, 18: 13, 19: 13, 20: 5, 21: 5, 22: 7, 23: 1, 24: 5, 26: 5, 27: 2, 30: 3, 31: 1, 32: 2, 37: 1}

per-bucket training-document counts:


,count,min,50%,max
bucket,,,,
frequent,34.0,1320.0,2524.5,11577.0
medium,33.0,540.0,666.0,1276.0
rare,33.0,270.0,367.0,524.0


most frequent 15 and rarest 15 statute ids:


,train_docs,train_share,dev_docs,bucket
label,,,,
5,11577,0.2736,2936,frequent
34,10304,0.2435,2596,frequent
97,7817,0.1847,1970,frequent
43,7665,0.1811,1932,frequent
39,7056,0.1667,1779,frequent
12,4288,0.1013,1086,frequent
13,4100,0.0969,1033,frequent
10,4077,0.0963,1033,frequent
77,3990,0.0943,1015,frequent


... (3 outputs left out)

' a  b  both  P(b|a)  P(a|b)   lift\n20 28   438   0.926   0.746 66.753\n15 16   277   0.824   0.417 52.458\n29 49   222   0.447   0.584 49.740\n49 50   142   0.374   0.447 49.725\n29 50   164   0.330   0.516 43.909\n64 86   228   0.362   0.628 42.187\n23 25    80   0.263   0.279 38.800\n29 36   245   0.493   0.431 36.724\n21 47   437   0.644   0.502 31.267\n21 53   625   0.920   0.494 30.815\n36 50   123   0.217   0.387 28.815\n47 53   712   0.817   0.563 27.366'

statutes that co-occur with the most others (hubs): [(0, 99), (1, 99), (7, 99), (5, 99), (12, 99)]


,docs,words_mean,words_median,words_p90,words_max
split,,,,,
train,42315,1126.5,432.0,2768.0,86858
dev,10181,1248.5,543.0,2957.0,69230
test,13019,1211.7,532.0,2811.4,134848


statutes/doc by length quartile (train): [3.78, 3.48, 3.55, 4.03]
share of documents that are lower-case: {'train': 0.0, 'dev': 0.0, 'test': 0.0}
=> lower-case all splits: False
placeholder tokens in the first 3000 training docs (masked entities): [('<ENTITY>', 178065), ('<SECTION>', 16293), ('<ACT>', 16166), ('[deceased]', 10), ('[or]', 4), ('[ALLOWED]', 3), ('[FIR]', 2), ('<br>', 2), ('[MD]', 2), ('[Namdeo]', 1)]

example (dev, first 700 chars):
 Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though the prosecution relied on the evidence of many persons who supposedly witnessedthe occurrence, while deposing in Court, most of them resiled from the statements madeduring investigation. The version projected by the prosecution was that the deceased wasthrottled and also poison was administered to him. But the report of the Forensic ScienceLaboratory stated that the viscera did <ENTITY> contain

## 4. Explicit statute mentions
If the judgments still name the provisions ("section 302", "u/s 34", "s. 120-b"), those numbers are a direct
signal. For every statute id we look for the section number whose mention is most *specific* to documents carrying
that id (lift = P(number | id) / P(number)), which also tells us what the anonymous ids mean.

In [7]:
SEC_RE = re.compile(r"(?:\bsections?|\bsecs?\.?|\bu/s\.?|\bs\.)\s*(\d{1,3}[a-z]?(?:\s*\(\s*\d+\s*\))?)", re.I)
def sec_numbers(t): return {re.sub(r"\s+", "", m.group(1).lower()) for m in SEC_RE.finditer(t)}
MENTIONS = {s: [sec_numbers(t) for t in d["texts"]] for s, d in DATA.items()}
for s in MENTIONS:
    print(f"{s}: documents mentioning at least one section number: {np.mean([bool(m) for m in MENTIONS[s]]):.1%} | "
          f"mean distinct numbers per doc: {np.mean([len(m) for m in MENTIONS[s]]):.2f}")
num_cnt = Counter(x for m in MENTIONS["train"] for x in m)
SEC_MAP = []
if num_cnt:
    nums = [x for x, c in num_cnt.items() if c >= 20]; nidx = {x: i for i, x in enumerate(nums)}
    M = np.zeros((len(MENTIONS["train"]), len(nums)), np.uint8)
    for i, m in enumerate(MENTIONS["train"]):
        for x in m:
            if x in nidx: M[i, nidx[x]] = 1
    joint = Y["train"].T.astype(np.int64) @ M.astype(np.int64)                 # (labels, numbers)
    p_num = M.mean(0) + 1e-9
    for l in range(N_LABELS):
        if cnt_tr[l] == 0 or not len(nums): continue
        cond = joint[l] / cnt_tr[l]; lift = cond / p_num; j = int(np.argmax(np.where(cond >= 0.3, lift, 0)))
        if cond[j] >= 0.3:
            SEC_MAP.append({"label": l, "train_docs": int(cnt_tr[l]), "section": nums[j], "P(mention|label)": cond[j],
                            "P(label|mention)": joint[l, j] / M[:, j].sum(), "lift": lift[j]})
    sm = pd.DataFrame(SEC_MAP)
    print(f"statute ids with a clearly associated section number: {len(sm)} of {N_LABELS}")
    if len(sm): display(sm.sort_values("train_docs", ascending=False).head(25).round(3).to_string(index=False))
    # how far would "predict the ids whose number is mentioned" get on dev? (a rule, not a model)
    if len(sm):
        num2labs = defaultdict(set)
        for r in SEC_MAP: num2labs[r["section"]].add(r["label"])
        P_rule = multihot([sorted({l for x in m for l in num2labs.get(x, ())}) for m in MENTIONS["dev"]])
        print(f"mention rule on dev: macro-F1 {f1_score(Y['dev'], P_rule, average='macro', zero_division=0):.4f}, "
              f"micro-F1 {f1_score(Y['dev'], P_rule, average='micro', zero_division=0):.4f}")
else:
    print("no section numbers found — statute mentions appear to be masked; the model must infer statutes from the facts")

train: documents mentioning at least one section number: 2.9% | mean distinct numbers per doc: 0.04
dev: documents mentioning at least one section number: 4.0% | mean distinct numbers per doc: 0.06
test: documents mentioning at least one section number: 4.0% | mean distinct numbers per doc: 0.06
statute ids with a clearly associated section number: 0 of 100


## 5. Tokenisation → compact token files
`normalize(text, "collapse_all")` (as in pretraining) → BPE ids; all documents of a split in one `uint16` file,
`offsets[i]:offsets[i+1]` = document i. Same format as Phase 4 (labels are lists of statute ids).

In [8]:
from tokenizers import Tokenizer
TOK_PATH = find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json")
assert TOK_PATH is not None, "attach the NB1 output (tokenizer/bpe32768/tokenizer.json) as an input"
TOK = Tokenizer.from_file(str(TOK_PATH)); print("tokenizer:", TOK_PATH, "| vocab", TOK.get_vocab_size())
assert TOK.get_vocab_size() <= 65535

TOKSTATS = {}
for s, d in DATA.items():
    t0 = time.time(); path = P5_DATA / f"{s}_tokens.bin"; offsets = [0]; n_tok = 0
    with open(str(path) + ".tmp", "wb") as f:
        for i in range(0, len(d["texts"]), 1000):
            encs = TOK.encode_batch([normalize(t, "collapse_all") for t in d["texts"][i:i + 1000]])
            for e in encs:
                a = np.asarray(e.ids, dtype=np.uint16); f.write(a.tobytes()); n_tok += len(a); offsets.append(n_tok)
    os.replace(str(path) + ".tmp", path)
    offsets = np.asarray(offsets, dtype=np.int64); L = np.diff(offsets)
    np.save(P5_DATA / f"{s}_offsets.npy", offsets)
    TOKSTATS[s] = {"docs": len(L), "tokens_total_M": n_tok / 1e6, "tokens_mean": L.mean(), "tokens_median": np.median(L),
                   "tokens_p90": np.percentile(L, 90), "tokens_max": int(L.max()), "> 512": (L > 512).mean(),
                   "> 1024": (L > 1024).mean(), "> 2048": (L > 2048).mean(), "> 4096": (L > 4096).mean(), "sec": time.time() - t0}
display(pd.DataFrame(TOKSTATS).T.round(3))
for s, d in DATA.items():
    save_json({"ids": d["ids"], "labels": d["labels"]}, P5_DATA / f"{s}_meta.json")

tokenizer: /kaggle/input/datasets/aaravoswal/cs781-p1-nb1/cs781_p1/tokenizer/bpe32768/tokenizer.json | vocab 32768


,docs,tokens_total_M,tokens_mean,tokens_median,tokens_p90,tokens_max,> 512,> 1024,> 2048,> 4096,sec
train,42315.0,72.169,1705.524,645.0,4168.0,139963.0,0.570,0.392,0.251,0.103,127.772
dev,10181.0,19.379,1903.411,804.0,4487.0,116158.0,0.620,0.444,0.289,0.117,34.626
test,13019.0,24.102,1851.328,797.0,4279.2,220420.0,0.619,0.442,0.274,0.108,43.438


## 6. Baselines on dev (official metric) — sections and decision rules
**Frequency prior**: every document gets the k most frequent statutes. **TF-IDF + logistic regression**: word
1–2-grams hashed into 2^19 features, one class-balanced binary classifier per statute (binary relevance), trained on
the first 512 words, the last 512 words or the whole document. Each probability matrix is turned into statute sets
with five rules:

| rule | how |
|---|---|
| `thr0.5` | every statute with p ≥ 0.5 |
| `global` | one threshold for all statutes, chosen on dev |
| `per_label` | one threshold per statute, chosen on dev (statutes with < 5 dev positives keep the global threshold) — scored **2-fold cross-fitted** on dev (thresholds from one half, score on the other) so the number is not optimistic |
| `topk` | the k most probable statutes (k ≈ mean statutes per document) |
| `+1` | `global`, plus the single most probable statute when nothing passes (every document has ≥ 1 statute) |

Metrics as in `evaluate_phase5.py`: macro-F1 (primary), micro-F1, subset accuracy; plus macro-F1 per frequency group.

In [9]:
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from joblib import Parallel, delayed

def scores(Yt, Yp):
    out = {"macro_f1": f1_score(Yt, Yp, average="macro", zero_division=0), "micro_f1": f1_score(Yt, Yp, average="micro", zero_division=0),
           "subset_acc": accuracy_score(Yt, Yp), "pred/doc": Yp.sum(1).mean()}
    per = f1_score(Yt, Yp, average=None, zero_division=0)
    for b in ("frequent", "medium", "rare"): out[f"F1_{b}"] = float(per[BUCKET == b].mean())
    return out

GRID = np.round(np.arange(0.02, 0.96, 0.02), 2)
def tune_global(Yt, P):
    s = [f1_score(Yt, (P >= t).astype(np.uint8), average="macro", zero_division=0) for t in GRID]
    return float(GRID[int(np.argmax(s))])
def tune_per_label(Yt, P, t_glob, min_pos=5):
    thr = np.full(P.shape[1], t_glob)
    for l in range(P.shape[1]):
        if Yt[:, l].sum() < min_pos: continue
        f = [f1_score(Yt[:, l], (P[:, l] >= t).astype(np.uint8), zero_division=0) for t in GRID]
        thr[l] = GRID[int(np.argmax(f))]
    return thr
def at_least_one(P, Yp):
    Yp = Yp.copy(); none = Yp.sum(1) == 0; Yp[none, P[none].argmax(1)] = 1; return Yp
def decision_rules(Yt, P, k):
    res = {"thr0.5": scores(Yt, (P >= 0.5).astype(np.uint8))}
    tg = tune_global(Yt, P); Yg = (P >= tg).astype(np.uint8); res[f"global@{tg:.2f}"] = scores(Yt, Yg)
    rng = np.random.default_rng(0); half = rng.permutation(len(Yt)) < len(Yt) // 2; Ycf = np.zeros_like(Yt)
    for a, b in ((half, ~half), (~half, half)):                       # tune on a, apply on b
        th = tune_per_label(Yt[a], P[a], tune_global(Yt[a], P[a])); Ycf[b] = (P[b] >= th).astype(np.uint8)
    res["per_label(cross-fit)"] = scores(Yt, Ycf)
    top = np.zeros_like(Yt); top[np.arange(len(P))[:, None], np.argsort(-P, 1)[:, :k]] = 1; res[f"top{k}"] = scores(Yt, top)
    res["global+1"] = scores(Yt, at_least_one(P, Yg))
    return res

def section(t, how, n=SECTION_WORDS):
    w = t.split()
    if how == "head": return " ".join(w[:n])
    if how == "tail": return " ".join(w[-n:])
    return " ".join(w[:20000])

K = max(1, int(round(card["train"].mean())))
BASE = {}
prior = np.tile(cnt_tr / len(Y["train"]), (len(Y["dev"]), 1))
for rule, r in decision_rules(Y["dev"], prior, K).items():
    if rule.startswith("top"): BASE[f"prior | {rule}"] = r
print(f"frequency prior, top{K}: macro-F1 {BASE[f'prior | top{K}']['macro_f1']:.4f}")

def fit_one(X, y, C=4.0):
    if y.sum() == 0: return None
    return LogisticRegression(C=C, max_iter=2000, class_weight="balanced", solver="liblinear").fit(X, y)

PROBS_DEV = {}
for how in BASELINE_SECTIONS:
    t0 = time.time()
    hv = HashingVectorizer(ngram_range=(1, 2), n_features=2 ** 19, alternate_sign=False, norm=None, dtype=np.float32)
    tf = TfidfTransformer(sublinear_tf=True)
    Xtr = tf.fit_transform(hv.transform(section(t, how) for t in DATA["train"]["texts"])).tocsr()
    Xdv = tf.transform(hv.transform(section(t, how) for t in DATA["dev"]["texts"])).tocsr()
    clfs = Parallel(n_jobs=N_JOBS)(delayed(fit_one)(Xtr, Y["train"][:, l]) for l in range(N_LABELS))
    P = np.zeros((Xdv.shape[0], N_LABELS), np.float32)
    for l, c in enumerate(clfs):
        if c is not None: P[:, l] = c.predict_proba(Xdv)[:, 1]
    PROBS_DEV[how] = P
    for rule, r in decision_rules(Y["dev"], P, K).items(): BASE[f"tfidf_{how} | {rule}"] = r
    print(f"tfidf {how:5s}: " + " | ".join(f"{rule.split(' | ')[1]} {r['macro_f1']:.4f}" for rule, r in BASE.items() if rule.startswith(f"tfidf_{how} ")) + f"  ({time.time()-t0:.0f}s)")
    del Xtr, Xdv, hv, tf, clfs; gc.collect()
base_df = pd.DataFrame(BASE).T
display(base_df.round(4))

# which statutes does the best baseline miss / over-predict? (full document, global rule)
P = PROBS_DEV["full"]; tg = tune_global(Y["dev"], P); Yp = (P >= tg).astype(np.uint8)
per = f1_score(Y["dev"], Yp, average=None, zero_division=0)
lab_df["dev_F1_tfidf_full"] = per[lab_df["label"].values]
print(f"F1 vs training frequency (tfidf full, global @ {tg:.2f}): Spearman ρ = {pd.Series(per).corr(pd.Series(cnt_tr.astype(float)), method='spearman'):.2f}")
print("worst 10 statutes with ≥ 20 dev documents:")
display(lab_df[lab_df["dev_docs"] >= 20].sort_values("dev_F1_tfidf_full").head(10).set_index("label").round(3))
fp = (Yp == 1) & (Y["dev"] == 0); fn = (Yp == 0) & (Y["dev"] == 1)
print("most false positives (label: count):", [(int(i), int(fp[:, i].sum())) for i in np.argsort(-fp.sum(0))[:8]])
print("most missed (label: count):", [(int(i), int(fn[:, i].sum())) for i in np.argsort(-fn.sum(0))[:8]])
# confusions: a missed gold statute a while a wrong statute b is predicted in the same document
conf = Counter()
for i in range(len(Yp)):
    for a in np.where(fn[i])[0]:
        for b in np.where(fp[i])[0]: conf[(int(a), int(b))] += 1
print("most frequent (missed a, predicted b instead) pairs:", conf.most_common(8))

frequency prior, top4: macro-F1 0.0150
tfidf head : thr0.5 0.3668 | global@0.46 0.3693 | per_label(cross-fit) 0.3713 | top4 0.3526 | global+1 0.3707  (1103s)
tfidf tail : thr0.5 0.3650 | global@0.46 0.3660 | per_label(cross-fit) 0.3677 | top4 0.3517 | global+1 0.3667  (1082s)
tfidf full : thr0.5 0.3935 | global@0.50 0.3935 | per_label(cross-fit) 0.3942 | top4 0.3733 | global+1 0.3944  (2373s)


,macro_f1,micro_f1,subset_acc,pred/doc,F1_frequent,F1_medium,F1_rare
prior | top4,0.0150,0.2341,0.0000,4.0000,0.0440,0.0000,0.0000
tfidf_head | thr0.5,0.3668,0.4768,0.0601,4.5903,0.4754,0.3391,0.2825
tfidf_head | global@0.46,0.3693,0.4742,0.0521,5.1181,0.4725,0.3459,0.2863
tfidf_head | per_label(cross-fit),0.3713,0.4722,0.0514,4.9530,0.4755,0.3492,0.2860
tfidf_head | top4,0.3526,0.4502,0.0121,4.0000,0.4375,0.3396,0.2781
tfidf_head | global+1,0.3707,0.4749,0.0544,5.1732,0.4730,0.3478,0.2881
tfidf_tail | thr0.5,0.3650,0.4678,0.0591,4.4950,0.4682,0.3342,0.2895
tfidf_tail | global@0.46,0.3660,0.4659,0.0526,5.0279,0.4668,0.3350,0.2932
tfidf_tail | per_label(cross-fit),0.3677,0.4610,0.0536,4.7733,0.4707,0.3401,0.2893
tfidf_tail | top4,0.3517,0.4402,0.0116,4.0000,0.4268,0.3361,0.2898


F1 vs training frequency (tfidf full, global @ 0.50): Spearman ρ = 0.53
worst 10 statutes with ≥ 20 dev documents:


,train_docs,train_share,dev_docs,bucket,dev_F1_tfidf_full
label,,,,,
8,349,0.008,92,rare,0.051
0,343,0.008,86,rare,0.104
86,363,0.009,92,rare,0.110
65,348,0.008,105,rare,0.132
76,524,0.012,142,rare,0.150
2,367,0.009,98,rare,0.150
82,601,0.014,156,medium,0.154
9,818,0.019,206,medium,0.169
98,490,0.012,123,rare,0.180


most false positives (label: count): [(97, 1406), (43, 1172), (5, 1135), (39, 1031), (12, 863), (96, 824), (13, 751), (51, 743)]
most missed (label: count): [(5, 999), (43, 860), (97, 757), (39, 647), (10, 512), (44, 488), (12, 429), (34, 415)]
most frequent (missed a, predicted b instead) pairs: [((5, 97), 116), ((12, 5), 108), ((13, 5), 94), ((43, 39), 90), ((43, 5), 88), ((39, 97), 88), ((14, 5), 82), ((39, 5), 80)]


## 7. Save

In [10]:
save_json([{"id": i, "gold": list(map(int, g))} for i, g in zip(DATA["dev"]["ids"], DATA["dev"]["labels"])], P5_DATA / "truth_dev.json")
np.save(P5_DATA / "cooccurrence_train.npy", C)
lab_df.to_csv(P5_DATA / "label_stats.csv", index=False)
meta = {"tokenizer": TOKENIZER, "n_labels": int(N_LABELS), "splits": {s: len(d["ids"]) for s, d in DATA.items()},
        "statutes_per_doc_mean": {s: float(card[s].mean()) for s in card}, "topk_default": K,
        "label_train_counts": cnt_tr.tolist(), "label_dev_counts": cnt_dv.tolist(), "label_bucket": BUCKET.tolist(),
        "lower_case_share": LOWER_SHARE, "lowercased": bool(DO_LOWER), "overlap_identical_texts": OVERLAP,
        "train_duplicates_removed": int(sum(len(g) - 1 for g in dups.values())), "train_duplicate_groups_with_different_sets": int(conflict),
        "section_number_map": SEC_MAP, "mention_share": {s: float(np.mean([bool(m) for m in MENTIONS[s]])) for s in MENTIONS},
        "tokens": {s: {k: float(v) for k, v in st.items()} for s, st in TOKSTATS.items()},
        "baselines_dev": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in BASE.items()}}
save_json(meta, P5_DATA / "meta.json")
print(json.dumps({k: v for k, v in meta.items() if k not in ("baselines_dev", "label_train_counts", "label_dev_counts", "label_bucket", "section_number_map")}, indent=1))
print("saved:", sorted(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in P5_DATA.iterdir()))

{
 "tokenizer": "bpe32768",
 "n_labels": 100,
 "splits": {
  "train": 42315,
  "dev": 10181,
  "test": 13019
 },
 "statutes_per_doc_mean": {
  "train": 3.7099137421718065,
  "dev": 3.9157253707887243
 },
 "topk_default": 4,
 "lower_case_share": {
  "train": 9.45291267871913e-05,
  "dev": 0.0,
  "test": 7.681081496274676e-05
 },
 "lowercased": false,
 "overlap_identical_texts": {
  "dev_in_train": 118,
  "test_in_train": 153,
  "test_in_dev": 98
 },
 "train_duplicates_removed": 435,
 "train_duplicate_groups_with_different_sets": 80,
 "mention_share": {
  "train": 0.029067706487061327,
  "dev": 0.03958353796287201,
  "test": 0.04047929948536754
 },
 "tokens": {
  "train": {
   "docs": 42315.0,
   "tokens_total_M": 72.169255,
   "tokens_mean": 1705.524164008035,
   "tokens_median": 645.0,
   "tokens_p90": 4168.0,
   "tokens_max": 139963.0,
   "> 512": 0.5698688408365827,
   "> 1024": 0.3923431407302375,
   "> 2048": 0.2514238449722321,
   "> 4096": 0.10280042538107054,
   "sec": 127.77225